In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
DATA_DIR = "../data/Mosquito_dataset"   
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.3, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int")

tmp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.3, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", shuffle=False)

Found 3000 files belonging to 3 classes.
Using 2100 files for training.
Found 3000 files belonging to 3 classes.
Using 900 files for validation.


In [4]:
n_batches = tmp_ds.cardinality().numpy()
test_ds = tmp_ds.take(n_batches // 2)
val_ds = tmp_ds.skip(n_batches // 2)


In [5]:
class_names = train_ds.class_names
print("Sınıflar:", class_names)

Sınıflar: ['AEDES', 'ANOPHELES', 'CULEX']


In [6]:
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


In [7]:
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [8]:
def build_cnn(num_classes):
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)   # piksel değerlerini [0,1]'e çek
    x = data_aug(x)

    for filters in [32, 64, 128, 128]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs)

model = build_cnn(len(class_names))
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 sequential (Sequential)     (None, 224, 224, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 224, 224, 32)      864       
                                                                 
 batch_normalization (BatchN  (None, 224, 224, 32)     128       
 ormalization)                                                   
                                                                 
 re_lu (ReLU)                (None, 224, 224, 32)      0         
                                                             

In [9]:
cbs = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Epoch 1/30
66/66 [==============================] - 147s 2s/step - loss: 0.2845 - accuracy: 0.8857 - val_loss: 2.7942 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 2/30
66/66 [==============================] - 128s 2s/step - loss: 0.1454 - accuracy: 0.9500 - val_loss: 4.3957 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 3/30
66/66 [==============================] - 131s 2s/step - loss: 0.1055 - accuracy: 0.9619 - val_loss: 5.9759 - val_accuracy: 0.0000e+00 - lr: 0.0010
Epoch 4/30
66/66 [==============================] - 133s 2s/step - loss: 0.0741 - accuracy: 0.9767 - val_loss: 5.7748 - val_accuracy: 0.0000e+00 - lr: 5.0000e-04
Epoch 5/30
66/66 [==============================] - 134s 2s/step - loss: 0.0798 - accuracy: 0.9757 - val_loss: 7.1041 - val_accuracy: 0.0000e+00 - lr: 5.0000e-04
Epoch 6/30
66/66 [==============================] - 133s 2s/step - loss: 0.0568 - accuracy: 0.9833 - val_loss: 5.0097 - val_accuracy: 0.0000e+00 - lr: 2.5000e-04


In [10]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")


14/14 [==============================] - 5s 363ms/step - loss: 2.7926 - accuracy: 0.0000e+00
Test accuracy: 0.0000


In [11]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)


14/14 [==============================] - 5s 343ms/step


In [12]:
model.save("mosquito_cnn.keras")

ValueError: Number of classes, 2, does not match size of target_names, 3. Try specifying the labels parameter